# Fase 0 — Extracción de datos de Firestore

Este notebook extrae la jerarquía completa de Firestore (`sessions → levels → rooms`)
y la guarda en `data/raw/` como JSON estructurado.

**Prerrequisitos:**
- `credentials/firebase-service-account.json` presente (no subir a git)
- `pip install cryptography` (única dependencia extra)

**Salidas:**
- `data/raw/sessions_raw.json` — jerarquía completa sin modificar
- `data/raw/extraction_metadata.json` — metadatos de la extracción (fecha, conteos)

## 0. Configuración

In [1]:
import sys
import json
import datetime
from pathlib import Path

# Añadir src/ al path para importar firestore_client
sys.path.insert(0, str(Path('..') / 'src'))

from firestore_client import FirestoreClient

# Rutas
CREDENTIALS_PATH = Path('..') / 'credentials' / 'firebase-service-account.json'
RAW_DIR          = Path('..') / 'data' / 'raw'
RAW_DIR.mkdir(parents=True, exist_ok=True)

print(f'Credenciales: {CREDENTIALS_PATH.resolve()}')
print(f'Salida raw:   {RAW_DIR.resolve()}')

Credenciales: D:\DataScience\Proyecto-TFM-HackAndSlash\credentials\firebase-service-account.json
Salida raw:   D:\DataScience\Proyecto-TFM-HackAndSlash\data\raw


## 1. Conexión y extracción

In [2]:
client = FirestoreClient(str(CREDENTIALS_PATH))
print(f'Proyecto Firebase: {client.get_project_id()}')

Proyecto Firebase: hackandslash-aabf7


In [3]:
print('Extrayendo sesiones de Firestore...')
sessions = client.export_all_sessions()
print(f'\n✓ Total sesiones extraídas: {len(sessions)}')

Extrayendo sesiones de Firestore...


  [20260512_132734_1952] 2 niveles, 11 salas


  [20260512_183848_6726] 1 niveles, 1 salas


  [20260512_184104_4372] 1 niveles, 1 salas


  [20260512_184136_8603] 4 niveles, 24 salas


...

  [20260610_063417_2182] 4 niveles, 24 salas


  [20260610_065821_1264] 1 niveles, 5 salas


  [20260610_070210_5789] 1 niveles, 4 salas


  [20260610_070710_4556] 4 niveles, 24 salas


  [20260610_072316_3553] 2 niveles, 6 salas

✓ Total sesiones extraídas: 58


## 2. Resumen de lo extraído

In [4]:
total_levels = sum(len(s['levels']) for s in sessions)
total_rooms  = sum(len(l['rooms']) for s in sessions for l in s['levels'])

print(f'Sesiones : {len(sessions)}')
print(f'Niveles  : {total_levels}')
print(f'Salas    : {total_rooms}')

# Versiones y plataformas presentes
versions  = sorted({s.get('gameVersion', '?') for s in sessions})
platforms = sorted({s.get('platform', '?') for s in sessions})
elements  = sorted({s.get('playerElement', '?') for s in sessions})

print(f'\nVersiones de juego : {versions}')
print(f'Plataformas        : {platforms}')
print(f'Elementos jugados  : {elements}')

victories = sum(1 for s in sessions if s.get('isVictory'))
print(f'\nVictorias: {victories}/{len(sessions)} ({victories/len(sessions)*100:.1f}%)')

Sesiones : 58
Niveles  : 119
Salas    : 597

Versiones de juego : ['0.1', '0.7', '1.0', '1.4', '1.5']
Plataformas        : ['Editor', 'WebGL']
Elementos jugados  : ['Earth', 'Fire', 'Water', 'Wind']

Victorias: 12/58 (20.7%)


## 3. Inspección de una sesión

In [5]:
# Primera sesión — vista completa
s0 = sessions[0]
print(f'--- Sesión: {s0["sessionId"]} ---')
print(f'Elemento  : {s0.get("playerElement")}')
print(f'Victoria  : {s0.get("isVictory")}')
print(f'Kills     : {s0.get("totalKills")}')
print(f'Muertes   : {s0.get("totalDeaths")}')
print(f'Tiempo(s) : {s0.get("totalTimeSecs")}')
print(f'Niveles completados: {s0.get("levelsCompleted")}')
print(f'Comentario: {s0.get("playerComment") or "(vacío)"}')

print(f'\nNiveles: {len(s0["levels"])}')
for lv in s0['levels']:
    print(f'  {lv["levelId"]} — kills:{lv.get("kills")} deaths:{lv.get("deaths")} time:{lv.get("timeSecs")}s | {len(lv["rooms"])} salas')
    for rm in lv['rooms']:
        dynamic = {k: v for k, v in rm.items() if k not in ('roomId','timeSecs','deaths','damageTaken','firstSpell')}
        print(f'    {rm["roomId"]} | firstSpell:{rm.get("firstSpell")} | {dynamic}')

--- Sesión: 20260512_132734_1952 ---
Elemento  : Fire
Victoria  : False
Kills     : 41
Muertes   : 0
Tiempo(s) : 211.1
Niveles completados: 2
Comentario: (vacío)

Niveles: 2
  Level1 — kills:16 deaths:0 time:68.9s | 5 salas
    Room_01 | firstSpell:Projectile | {'cast_Blast': 1, 'kills_Barbarian1Hand': 2, 'cast_Projectile': 4, 'status_Burn': 6, 'killedWith_Fireball': 2}
    Room_02 | firstSpell:Projectile | {'kills_RangerBow': 1, 'status_Burn': 11, 'kills_Barbarian1Hand': 2, 'killedWith_Fireball': 1, 'damage_Unknown': 20.0, 'killedWith_FireBlast_(1)': 2, 'cast_Projectile': 6, 'blocked_Barbarian1Hand': 2, 'cast_Blast': 3}
    Room_03 | firstSpell:Projectile | {'status_Burn': 11, 'damage_Knight1H': 24.0, 'killedWith_Fireball': 2, 'kills_Knight2H': 1, 'killedWith_FireBlast_(1)': 2, 'kills_Rogue': 1, 'damage_Barbarian1Hand': 24.0, 'cast_Projectile': 5, 'kills_Knight1H': 2, 'cast_Blast': 2}
    Room_04 | firstSpell:Projectile | {'kills_RangerBow': 1, 'cast_Beam': 1, 'kills_RangerCrossbow': 

## 4. Guardar en data/raw/

In [6]:
# Guardar jerarquía completa
output_path = RAW_DIR / 'sessions_raw.json'
with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(sessions, f, ensure_ascii=False, indent=2, default=str)

print(f'✓ Datos guardados en {output_path}')
print(f'  Tamaño: {output_path.stat().st_size / 1024:.1f} KB')

✓ Datos guardados en ..\data\raw\sessions_raw.json
  Tamaño: 430.6 KB


In [7]:
# Guardar metadatos de extracción
metadata = {
    'extracted_at'   : datetime.datetime.utcnow().isoformat() + 'Z',
    'project_id'     : client.get_project_id(),
    'total_sessions' : len(sessions),
    'total_levels'   : total_levels,
    'total_rooms'    : total_rooms,
    'game_versions'  : versions,
    'platforms'      : platforms,
    'elements'       : elements,
    'victories'      : victories,
    'defeat_rate'    : round(1 - victories / len(sessions), 4) if sessions else None,
}

meta_path = RAW_DIR / 'extraction_metadata.json'
with open(meta_path, 'w', encoding='utf-8') as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print(f'✓ Metadatos guardados en {meta_path}')
print(json.dumps(metadata, indent=2))

✓ Metadatos guardados en ..\data\raw\extraction_metadata.json
{
  "extracted_at": "2026-06-10T07:56:55.104173Z",
  "project_id": "hackandslash-aabf7",
  "total_sessions": 58,
  "total_levels": 119,
  "total_rooms": 597,
  "game_versions": [
    "0.1",
    "0.7",
    "1.0",
    "1.4",
    "1.5"
  ],
  "platforms": [
    "Editor",
    "WebGL"
  ],
  "elements": [
    "Earth",
    "Fire",
    "Water",
    "Wind"
  ],
  "victories": 12,
  "defeat_rate": 0.7931
}


C:\Users\Samuel\AppData\Local\Temp\claude\ipykernel_29024\2305853511.py:3: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  'extracted_at'   : datetime.datetime.utcnow().isoformat() + 'Z',


## 5. Inventario de campos dinámicos

Los campos de sala son dinámicos (dependen de qué enemigos/hechizos aparecen).
Este bloque los cataloga para planificar el preprocessing.

In [8]:
from collections import defaultdict

# Agrupar campos de sala por prefijo
prefix_counts = defaultdict(set)
static_fields = {'roomId', 'levelId', 'sessionId', 'timeSecs', 'deaths', 'damageTaken', 'firstSpell'}

for s in sessions:
    for lv in s['levels']:
        for rm in lv['rooms']:
            for k in rm:
                if k in static_fields:
                    continue
                prefix = k.split('_')[0] if '_' in k else k
                suffix = k[len(prefix)+1:] if '_' in k else ''
                prefix_counts[prefix].add(suffix)

print('Prefijos de campos dinámicos en salas:')
for prefix, values in sorted(prefix_counts.items()):
    print(f'  {prefix}_* → {sorted(values)[:8]}{" ..." if len(values) > 8 else ""}')

Prefijos de campos dinámicos en salas:
  blocked_* → ['Barbarian1Hand', 'Knight1H', 'KnightBossGold']
  cast_* → ['AOE', 'Aura', 'Beam', 'Blast', 'Projectile', 'Shield']
  damage_* → ['Barbarian1HBoss', 'Barbarian1Hand', 'Barbarian2HBoss', 'Barbarian2Hand', 'Knight1H', 'Knight2H', 'KnightBossBlack', 'KnightBossGold'] ...
  hit_* → ['AOE', 'Projectile']
  killedWith_* → ['AOE', 'Beam', 'BeamBody', 'Blast', 'EarthSlamSpikesAoe', 'FireBlast', 'FireBlast_(1)', 'Fireball'] ...
  kills_* → ['', 'Barbarian1HBoss', 'Barbarian1Hand', 'Barbarian2HBoss', 'Barbarian2Hand', 'Knight1H', 'Knight2H', 'KnightBossBlack'] ...
  miss_* → ['AOE', 'Projectile']
  movementDistance_* → ['']
  noMana_* → ['AOE', 'Aura', 'Beam', 'Blast', 'Projectile', 'Shield']
  spellDamage_* → ['AOE', 'Projectile']
  spellDamageOn_* → ['Barbarian1HBoss', 'Barbarian1Hand', 'Barbarian2HBoss', 'Barbarian2Hand', 'Knight1H', 'Knight2H', 'KnightBossBlack', 'KnightBossGold'] ...
  status_* → ['Burn', 'Knockback', 'Slow', 'Stun']
  t

---
**Siguiente paso:** `02_preprocessing.ipynb` — aplanar la jerarquía en DataFrames.